## load_zillow
Loads the three Zillow **long-form Parquet** datasets produced by `load_zillow_long` (`RAW_ZILLOW_LONG/<stem>_long/`) into three typed Bronze tables — `{BRONZE}.zillow_zhvi`, `_zori`, `_inventory` — one per feed in `ZILLOW_FEEDS`.

Unlike the CSV loaders this reads **Parquet** (already typed: `period_date DATE`, `value DOUBLE` — the documented exception to bronze-all-STRING), so there is no header validation. Source id columns are CamelCase in the Parquet (`RegionID`, ...) and are aliased to snake_case here. `source_file_path` carries no provenance in the Parquet, so it is injected as the logical wide-CSV path from `ZILLOW_FEEDS`.

**Write strategy (A):** MERGE on `(region_id, period_date)`; single `value` payload -> `UPDATE SET *` (no row_hash). **No archiving** (§18 deviation from §10 cell 7). DDL: `libs/ddl/bronze_ddl.py`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: BRONZE, AUDIT, RAW_ZILLOW, RAW_ZILLOW_LONG, ZILLOW_FEEDS,
# PIPELINE_RUN_ID, STATUS_*, StepLog, Utils, ingestion_log_insert, spark, dbutils, F.

STEP_SEQUENCE = 1                                   # position is owned by the orchestrator
SOURCE_SYSTEM = "zillow"

# Parquet id columns (CamelCase) -> Bronze snake_case. period_date/value pass through typed.
ID_ALIASES = {
    "RegionID":   "region_id",
    "SizeRank":   "size_rank",
    "RegionName": "region_name",
    "RegionType": "region_type",
    "StateName":  "state_name",
}
EXPECTED_PARQUET_COLS = list(ID_ALIASES) + ["period_date", "value"]

# ZILLOW_FEEDS: feed key -> source-file stem. Per feed:
#   long Parquet dir : f"{RAW_ZILLOW_LONG}{stem}_long"
#   target table     : f"{BRONZE}.zillow_{feed}"
#   source_file_path : f"{RAW_ZILLOW}{stem}.csv"  (logical wide-CSV provenance)
MERGE_KEYS = ["region_id", "period_date"]

In [ ]:
# Open the pipeline_step_log row (RUNNING). One row covers all three feeds.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = f"{BRONZE}.zillow_*",
)
print(f"load_zillow: step_log_id={step.step_log_id}")

In [ ]:
# Validate each feed's _long Parquet dir exists and has the expected columns (no CSV header
# check — Parquet is self-describing). Missing dirs => no_files exit. CHECK inside the try,
# EXIT outside (§10.1).
no_files = False
try:
    existing = {f.name.rstrip("/") for f in dbutils.fs.ls(RAW_ZILLOW_LONG)}
    missing, bad_cols = [], []
    for feed, stem in ZILLOW_FEEDS.items():
        if f"{stem}_long" not in existing:
            missing.append(f"{stem}_long")
            continue
        cols = spark.read.parquet(f"{RAW_ZILLOW_LONG}{stem}_long").columns
        if cols != EXPECTED_PARQUET_COLS:
            bad_cols.append((feed, cols))
    no_files = len(missing) == len(ZILLOW_FEEDS)        # nothing landed at all
    if missing and not no_files:
        raise FileNotFoundError(f"[zillow] missing _long dataset(s): {missing}")
    if bad_cols:
        raise ValueError(f"[zillow] unexpected Parquet columns (expected "
                         f"{EXPECTED_PARQUET_COLS}): {bad_cols}")
    if not no_files:
        print(f"load_zillow: {len(ZILLOW_FEEDS)} feed(s) validated.")
except Exception as e:
    step.fail(e); raise

if no_files:
    step.no_files()
    dbutils.notebook.exit(f"No Zillow _long datasets found at {RAW_ZILLOW_LONG}")

In [ ]:
# Read + shape + MERGE each feed into its own table. Aliases handle CamelCase -> snake_case;
# period_date/value pass through typed.
try:
    on_clause = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEYS)
    select_exprs = [F.col(src).alias(dst) for src, dst in ID_ALIASES.items()] + [
        F.col("period_date"), F.col("value")
    ]
    total_read = total_inserted = total_updated = 0
    processed_paths = []

    for feed, stem in ZILLOW_FEEDS.items():
        long_path = f"{RAW_ZILLOW_LONG}{stem}_long"
        wide_path = f"{RAW_ZILLOW}{stem}.csv"
        target    = f"{BRONZE}.zillow_{feed}"
        shaped_df = (
            spark.read.parquet(long_path).select(*select_exprs)
                .withColumn("source_file_path", F.lit(wide_path))
                .withColumn("inserted_ts", F.current_timestamp())
                .withColumn("run_id", F.lit(PIPELINE_RUN_ID))
        )
        n_read = shaped_df.count()
        shaped_df.createOrReplaceTempView("zillow_staging")

        pre_count = spark.table(target).count()
        metrics = spark.sql(f"""
            MERGE INTO {target} AS t
            USING zillow_staging AS s
            ON {on_clause}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """).first().asDict()
        post_count = spark.table(target).count()

        inserted = metrics.get("num_inserted_rows")
        if inserted is None:
            inserted = post_count - pre_count
        if post_count - pre_count != inserted:
            raise AssertionError(
                f"[{target}] Insert-count mismatch: MERGE reported {inserted:,} inserts, "
                f"row count grew by {post_count - pre_count:,}."
            )
        total_read     += n_read
        total_inserted += inserted
        total_updated  += metrics.get("num_updated_rows") or 0
        processed_paths.append(wide_path)
        print(f"load_zillow: {feed} -> {target}: read={n_read:,} inserted={inserted:,} "
              f"updated={metrics.get('num_updated_rows')}")

    step.rows_read    = total_read
    step.rows_written = total_inserted
    step.succeed()
    print(f"load_zillow: DONE read={total_read:,} inserted={total_inserted:,} "
          f"updated={total_updated:,}")
except Exception as e:
    step.fail(e); raise

# ingestion_log after succeed(), outside the try — one row per feed (logical CSV path).
files_df = spark.createDataFrame([(p,) for p in processed_paths], "source_file_path string")
res = ingestion_log_insert(
    spark, AUDIT, files_df, PIPELINE_RUN_ID, step.step_log_id,
    source_system=SOURCE_SYSTEM, target_table=f"{BRONZE}.zillow_*",
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"load_zillow: WARNING ingestion_log insert failed: {res['error_message']}")